## Objetivo da Análise

Esta análise busca responder:
“Quais fatores influenciam o preço e a demanda dos anúncios do Airbnb no Rio de Janeiro?”

In [ ]:
import pandas as pd

df = pd.read_csv("listings.csv")

df.head()

In [ ]:
cidade = "Rio de Janeiro"
print(f"Cidade analisada: {cidade}")

In [ ]:
#colunas, linhas
df.shape

In [ ]:

#colunas
df.columns
#Tipo de dado de cada coluna
df.dtypes

In [ ]:
#Ranking dos que possuem mais valores nulos
df.isnull().sum().sort_values(ascending=False)

In [ ]:
colunas_relevantes = [
    'price',
    'room_type', 
    'neighbourhood_cleansed',
    'accommodates',
    'number_of_reviews', 
    'reviews_per_month',  
    'amenities'                   
]

In [ ]:
df[colunas_relevantes].info()

### Tratamento de PRICE
**Justificativa de Price:** Realizei o tratamento dos dados na coluna PRICE, que possuia caracteres especiais. Os removi e converti a coluna em float para melhorar a ánalise com base nos preços

In [ ]:
df['price'].head(10)

In [ ]:
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)

In [ ]:
df['price'].head(10)

In [ ]:
df[colunas_relevantes].head(5)

In [ ]:
df['price'].isnull().sum()

In [ ]:
df = df.dropna(subset=['price']).copy()

In [ ]:
df['price'].isnull().sum()

### Transformação de Amenidades
**Justificativa das Amenidades:** Optei por extrair as 5 amenidades mais frequentes na base de dados pois elas representam a infraestrutura básica e o padrão mínimo esperado pela maioria dos hóspedes

In [ ]:
# 1. Removemos os colchetes e aspas da string usando regex
amenities_limpas = df['amenities'].str.replace(r'[\[\]"]', '', regex=True)

# 2. Separamos o texto por vírgulas para criar uma lista
amenities_listas = amenities_limpas.str.split(', ')

# 3.separa cada item da lista em uma linha temporária.
ranking_amenidades = amenities_listas.explode().value_counts()

# Mostrando amenidades que mais aparecem no Rio de Janeiro
display(ranking_amenidades.head(5))

In [ ]:
top_amenidades = ranking_amenidades.head(5).index.tolist()

import re
colunas_novas = []

for amenidade in top_amenidades:
    nome_limpo = re.sub(r'[^a-zA-Z0-9]', '_', amenidade.lower())
    nome_coluna = f'has_{nome_limpo}'
    
    df[nome_coluna] = df['amenities'].str.contains(amenidade, case=False, na=False).astype(int)
    
    colunas_novas.append(nome_coluna)

df[['price', 'accommodates'] + colunas_novas].head(10)    

In [ ]:
resumo = df.groupby('room_type').agg({
    'price': ['mean', 'median'],
    'number_of_reviews': 'mean',
    'accommodates': 'mean'
})
print(resumo)

In [ ]:
df.groupby('neighbourhood_cleansed')['price'].mean().sort_values(ascending=False).head(5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig1, axes1 = plt.subplots(1, 2, figsize=(14, 5))

# 1. Distribuição de preços
df_sem_outlier = df[df['price'] < df['price'].quantile(0.95)]
sns.histplot(df_sem_outlier['price'], bins=60, ax=axes1[0], color='steelblue')
axes1[0].set_title('Distribuição de Preços')
axes1[0].set_xlabel('Preço (R$)')

# 2. Preço médio por tipo de quarto
medias = df.groupby('room_type')['price'].median().sort_values()
medias.plot(kind='barh', ax=axes1[1], color='coral')
axes1[1].set_title('Preço Mediano por Tipo de Quarto')
axes1[1].set_xlabel('Preço Mediano (R$)')

plt.tight_layout()
plt.show() 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Média
top_mean = df.groupby('neighbourhood_cleansed')['price'].mean().nlargest(5)
top_mean.plot(kind='bar', ax=axes[0], color='orange')
axes[0].set_title('Top 5 — Média (com outliers)')
axes[0].tick_params(axis='x', rotation=45)

# Mediana
top_median = df.groupby('neighbourhood_cleansed')['price'].median().nlargest(5)
top_median.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Top 5 — Mediana')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
df_estacio = df[df['neighbourhood_cleansed'] == 'Estácio']

media_estacio = df_estacio['price'].mean()
mediana_estacio = df_estacio['price'].median()

moda_estacio = df_estacio['price'].mode()[0] 

print(f"--- Valores de Preço no Bairro Estácio ---")
print(f"Média (Average): R$ {media_estacio:.2f}")
print(f"Mediana (Valor do meio): R$ {mediana_estacio:.2f}")
print(f"Moda (Preço mais comum): R$ {moda_estacio:.2f}")

anuncios_caros = df_estacio.sort_values(by='price', ascending=False)

display(anuncios_caros[['id', 'name', 'room_type', 'price']].head(5))

In [ ]:
#Proxy de demanda = number_of_reviews
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x='price', y='number_of_reviews', ax=axes[0])
axes[0].set_xlim(0, 5000)
axes[0].set_title('Preço vs Demanda (até R$5000)')

sns.scatterplot(data=df, x='price', y='number_of_reviews', ax=axes[1])
axes[1].set_xlim(0, 1000)
axes[1].set_title('Preço vs Demanda (até R$1000)')

plt.tight_layout()
plt.show()

### Gráfico 1 - Distribuição de preços: 
A distribuição de preços é assimétrica à direita, com a maioria dos anúncios concentrada entre R$100 e R$500. O pico está  em torno de R$200-R$300, indicando que esse é o preço mais comum no mercado. A cauda longa à direita mostra a existência de imóveis de luxo que elevam a média, justificando o uso da mediana como medida mais representativa. (Caso utilize a média, a conta seria altamente influneciada)

### Gráfico 2 — Preço Mediano por Tipo de Quarto:
Hotel room apresenta o maior preço mediano (~R$450), seguido de Entire home/apt (R$350). Private room (R$200) e Shared room (R$100) são significativamente mais baratos. Isso reflete a lógica do mercado: acomodações completas e com serviços agregados comandam preços maiores, enquanto quartos compartilhados atendem um público mais sensível ao preço.

### Gráfico 3 — Top 5 Bairros por Preço Médio:
A média indica Estácio e Joá como os bairros mais caros, porém há distorção nos dados. No caso do Estácio, um valor extremamente alto (outlier) eleva artificialmente a média, não representando a realidade do bairro.

### Gráfico 3.1 — Top 5 Bairros por Preço mediano:
A mediana elimina o impacto de outliers e mostra valores mais realistas. Assim, os bairros apresentados refletem melhor o preço típico das diárias.

### Gráfico 4 - Demanda de procura
A quantidade de demanda está diretamente relacionada ao preço das diárias, a busca por diárias em valores mais acessíveis (R$150 -R$ 450) é muito maior do que para valores mais elevados

In [ ]:
from scipy import stats

# --- Teste 1: Amenidade (WiFi afeta número de reviews?) ---
# H0: número de reviews é igual com ou sem WiFi
# H1: anúncios com WiFi têm mais reviews

com_wifi = df[df['has_wifi'] == 1]['number_of_reviews']
sem_wifi = df[df['has_wifi'] == 0]['number_of_reviews']

stat, p = stats.mannwhitneyu(com_wifi, sem_wifi, alternative='greater')

print(f"--- Teste 1: Impacto do Wi-Fi na Demanda ---")
print(f"p-valor: {p:.4f}")

if p < 0.05:
    print("Conclusão: A diferença é REAL e comprovada!")
    print("Imóveis com Wi-Fi recebem significativamente mais avaliações (têm mais giro de hóspedes).")
else:
    print("Conclusão: NÃO existe diferença comprovada.")
    print("Ter Wi-Fi não garantiu mais avaliações neste cenário. Qualquer diferença vista é pura coincidência.")

In [ ]:
# --- Teste 2: Bairros têm preços diferentes? ---
# H0: preço médio é igual entre os bairros
# H1: pelo menos um bairro tem preço diferente

# Agrupando apenas bairros com um volume bom de anúncios (mais de 30)
grupos_bairros = [g['price'].values for _, g in df.groupby('neighbourhood_cleansed') if len(g) >= 30]

# Calculando a estatística
stat, p = stats.kruskal(*grupos_bairros)

print(f"\n--- Teste 2: Impacto da Localização no Preço ---")
print(f"p-valor: {p:.4f}")

if p < 0.05:
    print("Conclusão: O bairro afeta fortemente o valor da diária!")
    print("Existe uma diferença de preços real e estatisticamente comprovada entre as diferentes regiões.")
else:
    print("Conclusão: NÃO existe diferença de preço comprovada entre os bairros.")
    print("Estatisticamente, os anfitriões cobram valores parecidos independente da região analisada.")

In [ ]:
from scipy import stats

# --- Teste 3 ---
# Pergunta: Superhosts investem mais em Ar Condicionado do que anfitriões comuns?
# H0: Não existe relação
# H1: Existe relação

tabela = pd.crosstab(df['host_is_superhost'], df['has_air_conditioning'])
print("--- Tabela Cruzada (Contagem) ---")
print(tabela)
print("-" * 30)
stat, p, dof, expected = stats.chi2_contingency(tabela)

print(f"Teste 3 (Qui-Quadrado) | p-valor: {p:.4f}")
if p < 0.05:
    print("Conclusão: A relação é REAL e comprovada estatisticamente!")
    print("Anfitriões Superhost realmente têm um comportamento diferente e investem mais em Ar Condicionado.")
else:
    print("Conclusão: NÃO existe relação comprovada.")
    print("Qualquer diferença entre Superhosts e anfitriões comuns é puro acaso (coincidência).")

In [ ]:
#Difrença entre superhost e comuns
df['has_wifi'] = df['amenities'].str.contains('Wifi|Wi-Fi', case=False, na=False).astype(int)
df['has_air_cond'] = df['amenities'].str.contains('Air conditioning', case=False, na=False).astype(int)
df['has_kitchen'] = df['amenities'].str.contains('Kitchen', case=False, na=False).astype(int)
df['has_tv'] = df['amenities'].str.contains('TV', case=False, na=False).astype(int)
df['has_pool'] = df['amenities'].str.contains('Pool', case=False, na=False).astype(int)

resumo_host = df.groupby('host_is_superhost').agg(
    total_anuncios=('id', 'count'),
    preco_mediano=('price', 'median'), 
    reviews_medio=('number_of_reviews', 'mean'),
    
    perc_wifi=('has_wifi', 'mean'),
    perc_ar_cond=('has_air_cond', 'mean'),
    perc_cozinha=('has_kitchen', 'mean'),
    perc_tv=('has_tv', 'mean'),
    perc_piscina=('has_pool', 'mean')
)

colunas_perc = ['perc_wifi', 'perc_ar_cond', 'perc_cozinha', 'perc_tv', 'perc_piscina']
resumo_host[colunas_perc] = (resumo_host[colunas_perc] * 100).round(1)
resumo_host['preco_mediano'] = resumo_host['preco_mediano'].round(2)
resumo_host['reviews_medio'] = resumo_host['reviews_medio'].round(1)

display(resumo_host)